# Experiment No. 05 - VAE on MNIST

**CPU-friendly Jupyter notebook**

Run cells from top to bottom. These notebooks are designed for an Intel i7 college PC without a dedicated GPU. Some pretrained generative models remain too slow or memory-intensive for CPU; those experiments include lightweight demonstrations and clearly marked optional full-model sections.

## Aim
Train a small Variational Autoencoder on MNIST and generate digit-like images. CPU training is kept short.

In [ ]:
%pip install -q tensorflow matplotlib

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
(x_train, _), _ = tf.keras.datasets.mnist.load_data()
x_train = x_train[:10000].astype("float32").reshape(-1, 784) / 255.0
ds = tf.data.Dataset.from_tensor_slices(x_train).shuffle(5000).batch(128)

In [ ]:
latent_dim = 2
encoder = tf.keras.Sequential([
    tf.keras.layers.Input((784,)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(4)
])
decoder = tf.keras.Sequential([
    tf.keras.layers.Input((latent_dim,)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(784, activation="sigmoid")
])
opt = tf.keras.optimizers.Adam()

for epoch in range(3):
    for batch in ds:
        with tf.GradientTape() as tape:
            mu, log_var = tf.split(encoder(batch), 2, axis=1)
            z = mu + tf.exp(0.5 * log_var) * tf.random.normal(tf.shape(mu))
            reconstructed = decoder(z)
            rec = tf.reduce_sum(tf.keras.losses.binary_crossentropy(batch, reconstructed), axis=1)
            kl = -0.5 * tf.reduce_sum(1 + log_var - tf.square(mu) - tf.exp(log_var), axis=1)
            loss = tf.reduce_mean(rec + kl)
        variables = encoder.trainable_variables + decoder.trainable_variables
        gradients = tape.gradient(loss, variables)
        opt.apply_gradients(zip(gradients, variables))
    print("Epoch", epoch+1, "Loss:", float(loss))

In [ ]:
generated = decoder(tf.random.normal([10, 2])).numpy()
plt.figure(figsize=(10, 3))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(generated[i].reshape(28, 28), cmap="gray")
    plt.axis("off")
plt.show()

## Result
The VAE generated new MNIST-like digit images.